In [ ]:
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

In [ ]:
# Directory paths
drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/Colab Notebooks/coco_animal_images_dataset'

# Define data transforms and create a PyTorch dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset = ImageFolder(root=drive_dir, transform=transform)

In [ ]:
train_fraction = 0.7
val_fraction = 0.15
test_fraction = 0.15

In [ ]:
# Calculate the number of samples for each split
train_size = int(len(dataset) * train_fraction)
val_size = int(len(dataset) * val_fraction)
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

In [ ]:
# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Define ResNet50 model
class AnimalClassifier(nn.Module):
    def __init__(self, num_classes):
        super(AnimalClassifier, self).__init__()
        self.resnet = models.resnet50(pretrained=False)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

In [ ]:
# Initialize the model, loss function, and optimizer
num_classes = len(dataset.classes)
model = AnimalClassifier(num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
dataset.classes

['bear', 'cat', 'cow', 'dog', 'elephant', 'giraffe', 'horse', 'sheep', 'zebra']

In [ ]:
# Test evaluation function
def evaluate_test(model, data_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc='Testing', leave=False):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracy = correct / total
    print(f'Test Accuracy: {test_accuracy:.2%}')

In [ ]:
def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    print(f'Accuracy on evaluation set: {accuracy:.2%}')


In [ ]:
# Training loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 10
best_val_accuracy = 0.0

In [ ]:
scaler = GradScaler()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}', leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    print(f'Training Loss: {average_loss:.4f}')

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Validation', leave=False):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = correct / total
    print(f'Validation Accuracy: {val_accuracy:.2%}')

    # Save the model if it performs better on the validation set
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), '/content/drive/MyDrive/Colab Notebooks/resnet50_animal_classifier.pth')

print("Training complete.")

Training Loss: 1.9544


Validation Accuracy: 33.48%


Training Loss: 1.7590


Validation Accuracy: 33.74%


Training Loss: 1.6276


...

Validation Accuracy: 42.16%


Training Loss: 1.3705


Validation Accuracy: 48.17%


Training Loss: 1.3275


Validation:  55%|█████▌    | 54/98 [00:22<00:18,  2.35it/s]

In [ ]:
# Call the test evaluation function after training is complete
evaluate_test(model, test_loader)

In [ ]:
!wget https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/800px-Cat_November_2010-1a.jpg -O cat.jpg

--2024-03-11 09:04:19--  https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/800px-Cat_November_2010-1a.jpg
Resolving upload.wikimedia.org (upload.wikimedia.org)... 208.80.154.240, 2620:0:861:ed1a::2:b
Connecting to upload.wikimedia.org (upload.wikimedia.org)|208.80.154.240|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 237844 (232K) [image/jpeg]
Saving to: ‘cat.jpg’

cat.jpg             100%[===================>] 232.27K  --.-KB/s    in 0.05s   

2024-03-11 09:04:19 (4.77 MB/s) - ‘cat.jpg’ saved [237844/237844]



In [ ]:
image_path = '/content/cat.jpg'
image = Image.open(image_path)
input_tensor = transform(image)
input_batch = input_tensor.unsqueeze(0)
input_batch = input_batch.to(device)

In [ ]:
# Make the prediction
with torch.no_grad():
    output = model(input_batch)

In [ ]:
# Get the predicted class
_, predicted_class = torch.max(output, 1)

In [ ]:
# Map the class index to the class label
class_labels = dataset.classes  # Assuming you have access to the class labels
predicted_label = class_labels[predicted_class.item()]

print(f'The image is classified as: {predicted_label}')

The image is classified as: dog
